In [1]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 348896816910113977
, name: "/device:XLA_CPU:0"
device_type: "XLA_CPU"
memory_limit: 17179869184
locality {
}
incarnation: 9763436189731018699
physical_device_desc: "device: XLA_CPU device"
, name: "/device:XLA_GPU:0"
device_type: "XLA_GPU"
memory_limit: 17179869184
locality {
}
incarnation: 10664046200998896004
physical_device_desc: "device: XLA_GPU device"
, name: "/device:GPU:0"
device_type: "GPU"
memory_limit: 15881699328
locality {
  bus_id: 1
  links {
  }
}
incarnation: 13486390151140373238
physical_device_desc: "device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0"
]


In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cv2
from tqdm import tqdm
from glob import glob
from PIL import Image
from skimage.transform import resize
from sklearn.model_selection import train_test_split, KFold

import tensorflow as tf
import tensorflow.keras
from tensorflow.keras import backend as K
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# New codes  added
import torch

In [3]:
tf.__version__

'1.13.1'

## Building the training dataset.
Let's look at the train image list

In [4]:
img_path = "../input/isic2018/ISIC2018_Task1-2_Training_Input/ISIC2018_Task1-2_Training_Input/"
mask_path = '../input/isic2018/ISIC2018_Task1_Training_GroundTruth/ISIC2018_Task1_Training_GroundTruth/'

In [5]:
width = 128
height = 128
channels = 3

**Sort the file list in ascending order and seperate it into images and masks**<br/>
Each file has the form of either "subject_imageNum.tif" or "subject_imageNum_mask.tif", so we can extract `subject` and `imageNum` from each file name by using regular expression. `"[0-9]+"` means to find the first consecutive number.<br/>

In [6]:
train_img = glob(img_path + '*.jpg')
train_mask = [i.replace(img_path, mask_path).replace('.jpg', '_segmentation.png') for i in train_img]

        
print(train_img[:2],"\n" ,train_mask[:2])

['../input/isic2018/ISIC2018_Task1-2_Training_Input/ISIC2018_Task1-2_Training_Input/ISIC_0012706.jpg', '../input/isic2018/ISIC2018_Task1-2_Training_Input/ISIC2018_Task1-2_Training_Input/ISIC_0010192.jpg'] 
 ['../input/isic2018/ISIC2018_Task1_Training_GroundTruth/ISIC2018_Task1_Training_GroundTruth/ISIC_0012706_segmentation.png', '../input/isic2018/ISIC2018_Task1_Training_GroundTruth/ISIC2018_Task1_Training_GroundTruth/ISIC_0010192_segmentation.png']


### add read correctly with same lib

In [7]:
"""
# It contains 2594 training samples
img_files   = np.zeros([2594, height, width, channels])
mask_files   = np.zeros([2594, height, width])

print('Reading ISIC 2018')
for idx, (img_path, mask_path) in tqdm(enumerate(zip(train_img, train_mask))):
    img = cv2.imread(img_path)
    img = np.double(cv2.resize(img,(width,height)))
    img = img / 255
    img_files[idx, :,:,:] = img

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask,(width,height))
    mask = mask / 255
    mask[mask > 0.5] = 1
    mask[mask <= 0.5] = 0
    mask_files[idx, :,:] = mask    
         
print('Reading ISIC 2018 finished')
"""

"\n# It contains 2594 training samples\nimg_files   = np.zeros([2594, height, width, channels])\nmask_files   = np.zeros([2594, height, width])\n\nprint('Reading ISIC 2018')\nfor idx, (img_path, mask_path) in tqdm(enumerate(zip(train_img, train_mask))):\n    img = cv2.imread(img_path)\n    img = np.double(cv2.resize(img,(width,height)))\n    img = img / 255\n    img_files[idx, :,:,:] = img\n\n    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)\n    mask = cv2.resize(mask,(width,height))\n    mask = mask / 255\n    mask[mask > 0.5] = 1\n    mask[mask <= 0.5] = 0\n    mask_files[idx, :,:] = mask    \n         \nprint('Reading ISIC 2018 finished')\n"

Create data loader

# you should import the function from library which is used in the below that's the reason also you can not use pytorch data loader in tensorflow framework so we should have two data loader check the links and find the soulotion : https://www.tensorflow.org/guide/data      https://docs.pytorch.org/vision/main/datasets.html

In [8]:

class dataLoader(Dataset):
    def __init__(self, image_paths, mask_paths, width, height, transform=None):
        # We ONLY store the file paths (strings), not the actual images.
        # This takes almost 0 RAM.
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.width = width
        self.height = height
        self.transform = transform

    # Number of samples in dataset
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # ... (Load your images and masks like before) ...
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.width, self.height))
        image = image / 255.0
        
        mask_path = self.mask_paths[idx]
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.width, self.height))
        mask = mask / 255.0
        mask[mask > 0.5] = 1.0
        mask[mask <= 0.5] = 0.0
        
        # --- CHANGES START HERE ---
        
        # 1. DELETE this line (PyTorch specific):
        # image = np.transpose(image, (2, 0, 1)) 
        
        # 2. CHANGE this line for Mask:
        # Old (PyTorch): mask = np.expand_dims(mask, 0)  -> (1, 128, 128)
        # New (Keras):   mask = np.expand_dims(mask, axis=-1) -> (128, 128, 1)
        mask = np.expand_dims(mask, axis=-1)
        
        # 3. Return plain Numpy arrays (Keras handles them better than Torch tensors)
        # We assume the DataLoader will stack them later
        return image.astype(np.float32), mask.astype(np.float32)

# Starting from this point
print("Initializing Dataset...")

# Create the dataset object
train_dataset = dataLoader(train_img, train_mask, width, height)

# Create the Loader
# num_workers=2 means 2 CPU cores will load images in the background while GPU trains
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

print(f"Success! Dataset created with {len(train_dataset)} samples.")


NameError: name 'Dataset' is not defined

In [ ]:
# Display the first image and mask of the first subject.
image1 = np.array(Image.open(train_img[0]))
image1_mask = np.array(Image.open(train_mask[0]))
image1_mask = np.ma.masked_where(image1_mask == 0, image1_mask)

fig, ax = plt.subplots(1,3,figsize = (16,12))
ax[0].imshow(image1, cmap = 'gray')

ax[1].imshow(image1_mask, cmap = 'gray')

ax[2].imshow(image1, cmap = 'gray', interpolation = 'none')
ax[2].imshow(image1_mask, cmap = 'jet', interpolation = 'none', alpha = 0.7)

Now, I try to load all image files and store them variables X and y. Afther doing this, I recognize that it takes very much memory.<br/>
Please let me know if there are several efficient ways to store image file

## How to deal with train_masks.csv ?

Let's check that I did well

Let's modularize this work.

In [ ]:
from tensorflow.keras.models import Model, load_model
from tensorflow.keras import Input
from tensorflow.keras.layers import Input, Activation, BatchNormalization, Dropout, Lambda, Conv2D, Conv2DTranspose, MaxPooling2D, concatenate,add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
def dice_coef(y_true, y_pred):
    smooth = 0.0
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def iou(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum ( y_true_f * y_pred_f)
    union = K.sum ( y_true_f + y_pred_f - y_true_f * y_pred_f)
    return intersection/union


def dice_coef_loss(y_true, y_pred):
    return -dice_coef(y_true, y_pred)


In [ ]:
def unet(input_size=(256,256,1)):
    inputs = Input(input_size)
    
    conv1 = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    conv1 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    conv2 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    conv3 = Conv2D(256, (3, 3), activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D(pool_size=(2, 2))(conv3)

    conv4 = Conv2D(512, (3, 3), activation='relu', padding='same')(pool3)
    conv4 = Conv2D(512, (3, 3), activation='relu', padding='same')(conv4)
    pool4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    conv5 = Conv2D(1024, (3, 3), activation='relu', padding='same')(pool4)
    conv5 = Conv2D(1024, (3, 3), activation='relu', padding='same')(conv5)

    up6 = concatenate([Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(conv5), conv4], axis=3)
    conv6 = Conv2D(512, (3, 3), activation='relu', padding='same')(up6)
    conv6 = Conv2D(512, (3, 3), activation='relu', padding='same')(conv6)

    up7 = concatenate([Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(conv6), conv3], axis=3)
    conv7 = Conv2D(256, (3, 3), activation='relu', padding='same')(up7)
    conv7 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv7)

    up8 = concatenate([Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv7), conv2], axis=3)
    conv8 = Conv2D(128, (3, 3), activation='relu', padding='same')(up8)
    conv8 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv8)

    up9 = concatenate([Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv8), conv1], axis=3)
    conv9 = Conv2D(64, (3, 3), activation='relu', padding='same')(up9)
    conv9 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv9)

    conv10 = Conv2D(1, (1, 1), activation='sigmoid')(conv9)

    return Model(inputs=[inputs], outputs=[conv10])

In [ ]:
def keras_bridge(pytorch_loader):
    """
    Takes a PyTorch DataLoader and makes it look like a Keras Generator
    """
    while True:
        for images, masks in pytorch_loader:
            # PyTorch automatically converts to Tensor, so we convert back to Numpy for Keras
            yield images.numpy(), masks.numpy()

This part of code generated by AI. The original cell of training can not get the dataloader by PyTorch as input.

In [ ]:
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader

# 1. SETUP: Prepare the file lists (We use the paths, NOT the loaded images)
# Convert lists to numpy arrays so we can easily slice them with indices
all_img_paths = np.array(train_img) 
all_mask_paths = np.array(train_mask)

kf = KFold(n_splits=5, shuffle=False)

histories = []
losses = []
accuracies = []
dicecoefs = []
ious = []

EPOCHS = 10
BATCH_SIZE = 16

# Loop through the 5 folds
for k, (train_index, test_index) in enumerate(kf.split(all_img_paths)):
    print(f"--- Starting Fold {k+1} / 5 ---")
    
    # 2. SPLIT: Get the specific filenames for this fold
    X_train_paths = all_img_paths[train_index]
    y_train_paths = all_mask_paths[train_index]
    X_test_paths = all_img_paths[test_index]
    y_test_paths = all_mask_paths[test_index]
    
    # 3. LOADERS: Create the PyTorch DataLoaders for this specific fold
    # 'dataLoader' is the class you modified in Step 1
    train_ds = dataLoader(X_train_paths, y_train_paths, width, height)
    val_ds = dataLoader(X_test_paths, y_test_paths, width, height)
    
    # num_workers=2 uses parallel processing (make sure to set this!)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # 4. MODEL: Define and Compile
    model = unet(input_size=(height, width, channels))
    model.compile(optimizer=Adam(lr=5e-6), loss=dice_coef_loss, 
                  metrics=[iou, dice_coef, 'binary_accuracy'])

    model_checkpoint = ModelCheckpoint(str(k+1) + '_unet_skin_seg.hdf5', 
                                       verbose=1, 
                                       save_best_only=True)

    # 5. TRAIN: Use the 'keras_bridge' to feed data
    # Note: We must specify 'steps_per_epoch' because generators don't have a length Keras can see easily
    history = model.fit(
        keras_bridge(train_loader),
        steps_per_epoch=len(train_loader),
        epochs=EPOCHS,
        callbacks=[model_checkpoint],
        validation_data=keras_bridge(val_loader),
        validation_steps=len(val_loader)
    )
    
    # 6. EVALUATE
    # Reload best model
    model = load_model(str(k+1) + '_unet_skin_seg.hdf5', 
                       custom_objects={'dice_coef_loss': dice_coef_loss, 'iou': iou, 'dice_coef': dice_coef})
    
    # Use the bridge for evaluation too
    results = model.evaluate(keras_bridge(val_loader), steps=len(val_loader))
    results = dict(zip(model.metrics_names, results))
    
    histories.append(history)
    accuracies.append(results['binary_accuracy'])
    losses.append(results['loss'])
    dicecoefs.append(results['dice_coef'])
    ious.append(results['iou'])

print("Training Complete!")

In [ ]:
"""
kf = KFold(n_splits = 5, shuffle=False)

histories = []
losses = []
accuracies = []
dicecoefs = []
ious = []

EPOCHS = 10
BATCH_SIZE = 16

mask_files = mask_files[:, :, :, np.newaxis]

for k, (train_index, test_index) in enumerate(kf.split(img_files, mask_files)):
    X_train = img_files[train_index]
    y_train = mask_files[train_index]
    X_test = img_files[test_index]
    y_test = mask_files[test_index]
    
    model = unet(input_size=(height,width, channels))
    model.compile(optimizer=Adam(lr=5e-6), loss=dice_coef_loss, \
                      metrics=[iou, dice_coef, 'binary_accuracy'])

    model_checkpoint = ModelCheckpoint(str(k+1) + '_unet_skin_seg.hdf5', 
                                       verbose=1, 
                                       save_best_only=True)

    history = model.fit(X_train,
                        y_train,
                        epochs=EPOCHS,
                        callbacks=[model_checkpoint],
                        validation_data = (X_test, y_test))
    
    model = load_model(str(k+1) + '_unet_skin_seg.hdf5', custom_objects={'dice_coef_loss': dice_coef_loss, 'iou': iou, 'dice_coef': dice_coef})
    
    results = model.evaluate(X_test, y_test)
    results = dict(zip(model.metrics_names,results))
    
    histories.append(history)
    accuracies.append(results['binary_accuracy'])
    losses.append(results['loss'])
    dicecoefs.append(results['dice_coef'])
    ious.append(results['iou'])
    """

In [ ]:
import pickle

for h, history in enumerate(histories):

    keys = history.history.keys()
    fig, axs = plt.subplots(1, len(keys)//2, figsize = (25, 5))
    fig.suptitle('No. ' + str(h+1) + ' Fold Results', fontsize=30)

    for k, key in enumerate(list(keys)[:len(keys)//2]):
        training = history.history[key]
        validation = history.history['val_' + key]

        epoch_count = range(1, len(training) + 1)

        axs[k].plot(epoch_count, training, 'r--')
        axs[k].plot(epoch_count, validation, 'b-')
        axs[k].legend(['Training ' + key, 'Validation ' + key])
    
    with open(str(h+1) + '_skin_trainHistoryDict', 'wb') as file_pi:
        pickle.dump(history.history, file_pi)

In [ ]:
print('accuracies : ', accuracies)
print('losses : ', losses)
print('dicecoefs : ', dicecoefs)
print('ious : ', ious)

print('-----------------------------------------------------------------------------')
print('-----------------------------------------------------------------------------')

print('average accuracy : ', np.mean(np.array(accuracies)))
print('average loss : ', np.mean(np.array(losses)))
print('average dicecoefs : ', np.mean(np.array(dicecoefs)))
print('average ious : ', np.mean(np.array(ious)))
print()

print('standard deviation of accuracy : ', np.std(np.array(accuracies)))
print('standard deviation of loss : ', np.std(np.array(losses)))
print('standard deviation of dicecoefs : ', np.std(np.array(dicecoefs)))
print('standard deviation of ious : ', np.std(np.array(ious)))

In [ ]:
selector = np.argmin(abs(np.array(ious) - np.mean(ious)))
model = load_model(str(selector+1) + '_unet_skin_seg.hdf5', custom_objects={'dice_coef_loss': dice_coef_loss, 'iou': iou, 'dice_coef': dice_coef})

In [ ]:
for i in range(20):
    index=np.random.randint(0,len(img_files))
    print(i+1, index)
    img = cv2.imread(img_files[index])
    img = cv2.resize(img, (height, width))
    img = img[np.newaxis, :, :, :]
    img = img / 255
    pred = model.predict(img)

    plt.figure(figsize=(12,12))
    plt.subplot(1,3,1)
    plt.imshow(np.squeeze(img))
    plt.title('Original Image')
    plt.subplot(1,3,2)
    plt.imshow(np.squeeze(cv2.resize(cv2.imread(mask_files[index]), (height, width))))
    plt.title('Original Mask')
    plt.subplot(1,3,3)
    plt.imshow(np.squeeze(pred) > .5)
    plt.title('Prediction')
    plt.show()